In [ ]:
from SFS.src.py.utils import *
from numpy.fft import fft2, ifft, ifft2, fft, fftfreq, fftshift, ifftshift, rfftfreq, rfft2
from IPython.display import HTML

In [ ]:
number  = 6
m       = 6
folder  = "data/SETD_paper/{n}/{m}/".format(n=number,m=m)
para_folder  = "data/SETD_paper/{n}/{m}/1/".format(n=number,m=1)
num     = count_files(folder)
print(num)

In [ ]:
vid_notebook(folder, 0, skip=1, size=2)

In [ ]:
def get_K(field, q, con):
    r, u = con["r"], con["u"]
    phiq = 1j*q * ifft2(field)
    phi = fft2(phiq)
    return u * phi**2 / 2

In [ ]:
fn = "varphi"
field_av = get_field(folder + "{m}/".format(m = 1), fn)
for i in range(1, num):
    field_av += get_field(folder + "{m}/".format(m=i+1), fn)
field_av = field_av / num
field_av = np.mean(field_av, axis=1)[:, None]

fig, ax = plt.subplots(figsize=(5,3)) 
plt.plot(field_av[:,0])

In [ ]:
run_folder = folder + "{m}/".format(m = 3)
field = get_field(run_folder, fn) - field_av
param = get_para_folder_all(folder, para_folder=para_folder)
run_folder = folder+"{}/".format(1)
X, d, N, L, T, dt, con = param
a, figv = anim_fields_array([field,], param, skip=1, size=2, interval=50)
figv.tight_layout()
display(HTML(a.to_jshtml()))
plt.close(figv)

In [ ]:
print(L, N, T, con["u"])

In [ ]:
#! check this....

def get_w2(seed, folder, start=1):
    run_folder = folder + "{m}/".format(m = seed)
    # Is that right folder?
    X, d, N, L, T, dt, con = get_para_folder_all(folder, para_folder=para_folder)
    time = get_time(run_folder)

    field = get_field(run_folder, fn)
    nt, nx = np.shape(field)
    lnt = int(np.log2(nt))
    lnx = int(np.log2(nx))
    w2 = np.zeros((lnt-start, lnx-start))

    for i in range(start,lnt): # Time step
        ii = 2**(i+1)
        for j in range(start,lnx):
            jj = 2**(j+1)   # points in window
            nw = nx//jj     # Number of windows in the system
            for k in range(nw):
                i1, i2 = k*jj, (k+1)*jj
                htx = field[ii, i1:i2]
                w2[i-start,j-start] += (np.mean(htx**2) - np.mean(htx)**2) / nw

    xx = np.array([X[0][2**(j+1)-1] for j in range(start,lnx)])
    tt = np.array([time[2**(i+1)] for i in range(start,lnt)])

    return w2, xx, tt

In [ ]:
w2, xx, tt = get_w2(1, folder)
for i in range(1, num): w2 += get_w2(i+1, folder)[0]
w = np.sqrt(w2 / num)
nt, nx = np.shape(w)
print(nt,nx)
print(tt)

In [ ]:
NN = None
MM = None

print(tt)

b, c = np.polyfit(np.log(tt[MM:NN]), np.log(w[MM:NN,-1]), 1)
fig, ax = plt.subplots()
ax.plot(tt, np.exp(c)*tt**b, label='$\\beta = {b:.3f}$'.format(b=b))
ax.loglog(tt, w[:, -1], 'xk', label='$\\ell = L_{\\mathrm{sys}}$')
ax.legend()
ax.set_xlabel("$t$")
ax.set_ylabel("$w_\\ell(t)$")
ax.set_yticks([], minor=True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from SFS.src.py.utils import get_field, get_para

# Calculate the ensemble average of the mean height, spatial correlation, and variance at final time
def get_weak_metrics(folder_num, M, n_ens):
    # Extract the true dt_0 from the first run's parameters
    _, _, _, _, _, dt0, _ = get_para(f"data/SETD_paper/{folder_num}/1/1/")
    dt_vals = [dt0 / (2**(m-1)) for m in range(1, M+1)]
    
    mean_h_m, corr_m, var_m = [], [], []
    
    for m in range(1, M+1):
        h_seed, corr_seed, var_seed = [], [], []
        for seed in range(1, n_ens+1):
            base_folder = f"data/SETD_paper/{folder_num}/{m}/{seed}/"
            try: field = get_field(base_folder, "varphi")[-1]
            except: print(f"Can't retrieve {seed}"); continue
            h_seed.append(np.mean(field))
            C_dx = np.mean(field * np.roll(field, 1))
            corr_seed.append(C_dx)
            var_seed.append(np.var(field))
            
        mean_h_m.append(np.mean(h_seed))
        corr_m.append(np.mean(corr_seed))
        var_m.append(np.mean(var_seed))
        
    return dt_vals, [np.array(mean_h_m), np.array(corr_m), np.array(var_m)]

# Setup schemes to compare
schemes = [
    {'folder': number+0, 'label': 'ETD1', 'marker': '^'},
    {'folder': number+1, 'label': 'ETD2', 'marker': 'o'},
    {'folder': number+2, 'label': 'IF',   'marker': 's'},
]

# Read M and n_ens automatically
M = len([name for name in os.listdir(f"data/SETD_paper/{number}")])
n_ens = len([name for name in os.listdir(f"data/SETD_paper/{number}/1")])
# n_ens = 2**10
# M = 6
print(f"Read from data: M={M}, n_ens={n_ens}")

# Collect all data
datas = [[], [], []]
for i, s in enumerate(schemes):
    dt_vals, data = get_weak_metrics(s['folder'], M, n_ens)
    for i, d in enumerate(data):
        datas[i].append(d)

In [ ]:
metrics_to_plot = (
    ('\\varphi(t)', datas[0]),
    ('C(\\mathrm{d}x)', datas[1]),
    ('(\\varphi(t)-\\bar\\varphi(t))^2', datas[2]),
)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
titles = ["$\\bar \\varphi$", '$C(\\mathrm{d}x)$', '$\\mathrm{Var}(\\varphi)$']

for j, ax  in enumerate(axes):
    (var, data) = metrics_to_plot[j]
    for i, s in enumerate(schemes):
        ax.semilogx(dt_vals, data[i], marker=s['marker'], linestyle='-', label=s['label'])
    
    ax.set_xlabel(r'$\Delta t$')
    ax.set_ylabel(f'$E[{var}]$')
    ax.set_title(f'{titles[j]}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
refs = [d[1][-1] for d in datas]
errs = [[np.abs(dj[:-1] - refs[i]) for dj in d] for i, d in enumerate(datas)]

dt_plot = dt_vals[:-1]
dt_ref = np.array(dt_plot)

for i, err in enumerate(errs):
    plt.figure(figsize=(4, 3))
    for j, s in enumerate(schemes):
        plt.loglog(dt_plot, err[j], marker=s['marker'], linestyle='-', label=s['label'])

    plt.loglog(dt_ref, dt_ref * err[j][0]/dt_ref[0], 'k--', label=r'$\mathcal{O}(\Delta t)$')
    plt.loglog(dt_ref, dt_ref**2 * err[j][0]/dt_ref[0], 'k--', label=r'$\mathcal{O}(\Delta t^2)$') 
    plt.xlabel(r'$\Delta t$')
    plt.title(titles[i])
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()